In [1]:
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter,
    FieldCondition, MatchValue
)

load_dotenv()

PROCESSED_DATA = Path("../data/processed")
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # fast, good quality, 384 dimensions
QDRANT_URL = "http://localhost:6333"

print(f"Qdrant URL: {QDRANT_URL}")
print(f"Embedding model: {EMBEDDING_MODEL}")

Qdrant URL: http://localhost:6333
Embedding model: all-MiniLM-L6-v2


In [2]:
merged_df = pd.read_csv(PROCESSED_DATA / "merged_uci_races.csv")
course_data = pd.read_csv(PROCESSED_DATA / "course_data_clean.csv")

print(f"merged_df:   {merged_df.shape[0]:,} rows x {merged_df.shape[1]} cols")
print(f"course_data: {course_data.shape[0]:,} rows x {course_data.shape[1]} cols")

merged_df:   140,573 rows x 44 cols
course_data: 963 rows x 27 cols


In [3]:
client = QdrantClient(url=QDRANT_URL)

# Verify connection
collections = client.get_collections()
print(f"Existing collections: {[c.name for c in collections.collections]}")

Existing collections: []


In [4]:
model = SentenceTransformer(EMBEDDING_MODEL)

# Quick sanity check
test_embedding = model.encode("Tour de France Stage 17 mountain stage")
print(f"Embedding dimensions: {len(test_embedding)}")
print(f"Sample values: {test_embedding[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions: 384
Sample values: [-0.01837083  0.1146594   0.02340588 -0.08959314 -0.00486498]


In [5]:
def serialize_course_doc(row: pd.Series) -> str:
    """
    Convert a course_data row into a natural language document
    suitable for embedding. Numbers alone embed poorly — context matters.
    """
    parts = []

    # Race identity
    parts.append(f"Race: {row['Race Name']}.")

    # Distance
    if pd.notna(row.get('Distance')):
        parts.append(f"Distance: {row['Distance']:.1f} km.")

    # Elevation profile
    if pd.notna(row.get('Vertical Gain')) and pd.notna(row.get('Highest Elevation')):
        vg = row['Vertical Gain']
        he = row['Highest Elevation']
        le = row.get('Lowest Elevation', 0)
        ng = row.get('Net Gain', 0)

        if vg > 4000:
            stage_type = "a high mountain stage"
        elif vg > 2000:
            stage_type = "a hilly stage with significant climbing"
        elif vg > 800:
            stage_type = "a moderately hilly stage"
        else:
            stage_type = "a flat stage suited to sprinters"

        parts.append(
            f"This is {stage_type} with {vg:.0f}m of vertical gain, "
            f"reaching a maximum elevation of {he:.0f}m "
            f"and a minimum elevation of {le:.0f}m. "
            f"Net elevation change: {ng:.0f}m."
        )

    # Surface composition
    surface_parts = []
    surfaces = {
        'Asphalt': 'asphalt',
        'Road': 'road',
        'Cobblestones': 'cobblestones',
        'Compacted Gravel': 'compacted gravel',
        'Unpaved': 'unpaved sections',
        'Paved': 'paved road',
    }
    for col, label in surfaces.items():
        val = row.get(col)
        if pd.notna(val) and float(val) > 0.5:
            surface_parts.append(f"{val:.1f}km of {label}")

    if surface_parts:
        parts.append(f"Surface breakdown: {', '.join(surface_parts)}.")

    # Cobblestone flag — important for classics
    cob = row.get('Cobblestones')
    if pd.notna(cob) and float(cob) > 0:
        parts.append(
            f"Contains {cob:.1f}km of cobblestones — "
            f"a significant factor for classics specialists."
        )

    # Downhill
    dh = row.get('Downhill')
    if pd.notna(dh) and float(dh) > 0:
        parts.append(f"Total descending: {dh:.0f}m.")

    return " ".join(parts)

# Test on a known stage
tdf_stage17 = course_data[
    course_data['Race Name'] == '2023 Tour de France Stage 17'
]
if not tdf_stage17.empty:
    print("=== TDF 2023 Stage 17 (Queen Stage) ===")
    print(serialize_course_doc(tdf_stage17.iloc[0]))
    print()

# Test on a flat stage
tdf_stage21 = course_data[
    course_data['Race Name'] == '2023 Tour de France Stage 21'
]
if not tdf_stage21.empty:
    print("=== TDF 2023 Stage 21 (Champs-Elysees) ===")
    print(serialize_course_doc(tdf_stage21.iloc[0]))

=== TDF 2023 Stage 17 (Queen Stage) ===
Race: 2023 Tour de France Stage 17. Distance: 165.8 km. This is a high mountain stage with 5570m of vertical gain, reaching a maximum elevation of 2299m and a minimum elevation of 485m. Net elevation change: 1000m. Surface breakdown: 164.0km of asphalt, 123.0km of road, 1.2km of paved road. Total descending: 4570m.

=== TDF 2023 Stage 21 (Champs-Elysees) ===
Race: 2023 Tour de France Stage 21. Distance: 115.0 km. This is a moderately hilly stage with 820m of vertical gain, reaching a maximum elevation of 175m and a minimum elevation of 29m. Net elevation change: -90m. Surface breakdown: 69.6km of asphalt, 31.1km of road, 38.9km of cobblestones, 2.1km of compacted gravel, 2.8km of paved road. Contains 38.9km of cobblestones — a significant factor for classics specialists. Total descending: 910m.


In [6]:
def serialize_rider_doc(rider_name: str, year: int, df: pd.DataFrame) -> str:
    """
    Build a natural language summary of a rider's season
    from their race results for that year.
    """
    rider_df = df[
        (df['Name'] == rider_name) &
        (df['Year_results'] == year)
    ].copy()

    if rider_df.empty:
        return ""

    parts = []
    parts.append(f"Rider: {rider_name}. Season: {year}.")

    # Team
    team = rider_df['Team'].iloc[0]
    if pd.notna(team):
        parts.append(f"Team: {team}.")

    # Race count
    races = rider_df['Race_results'].unique()
    parts.append(f"Competed in {len(races)} races.")

    # Wins and podiums
    wins = rider_df[rider_df['Rank'] == 1]
    podiums = rider_df[rider_df['Top3'] == 1]
    top10s = rider_df[rider_df['Top10'] == 1]
    dnfs = rider_df[~rider_df['Did_Finish']]

    parts.append(
        f"Results: {len(wins)} wins, "
        f"{len(podiums)} podiums, "
        f"{len(top10s)} top-10 finishes, "
        f"{len(dnfs)} DNFs/DNS."
    )

    # List wins explicitly
    if not wins.empty:
        win_names = wins['Race Name'].tolist()
        parts.append(f"Won: {', '.join(win_names[:5])}.")

    # Best results outside wins
    top_results = rider_df[
        (rider_df['Rank'] > 1) & (rider_df['Did_Finish'])
    ].nsmallest(3, 'Rank')

    if not top_results.empty:
        best = [
            f"{row['Race Name']} (P{int(row['Rank'])})"
            for _, row in top_results.iterrows()
        ]
        parts.append(f"Other notable results: {', '.join(best)}.")

    # Stage race performance
    # gc_races = ['Tour de France', 'Giro d\'Italia', 'Vuelta a Espana']
    gc_races = ['Tour de France', 'Giro d', 'Vuelta a Espana']
    for gc in gc_races:
        gc_results = rider_df[rider_df['Race_results'].str.contains(gc, na=False)]
        if not gc_results.empty:
            best_gc = gc_results[gc_results['Did_Finish']]['Rank'].min()
            if best_gc < 999:
                parts.append(f"{gc}: best result P{int(best_gc)}.")

    return " ".join(parts)

# Test with a top rider
print("=== POGAČAR Tadej 2023 ===")
pog_doc = serialize_rider_doc("POGAČAR Tadej", 2023, merged_df)
print(pog_doc)

print("\n=== VINGEGAARD Jonas 2023 ===")
ving_doc = serialize_rider_doc("VINGEGAARD Jonas", 2023, merged_df)
print(ving_doc)

print("\n=== EVENEPOEL Remco 2023 ===")
even_doc = serialize_rider_doc("EVENEPOEL Remco", 2023, merged_df)
print(even_doc)

=== POGAČAR Tadej 2023 ===
Rider: POGAČAR Tadej. Season: 2023. Team: UAE Team Emirates. Competed in 7 races. Results: 9 wins, 14 podiums, 17 top-10 finishes, 1 DNFs/DNS. Won: 2023 Amstel Gold Race, 2023 Il Lombardia, 2023 La Fleche Wallonne, 2023 Paris-Nice Stage 4, 2023 Paris-Nice Stage 7. Other notable results: 2023 Tour de France Stage 14 (P2), 2023 Tour de France Stage 16 (P2), 2023 Tour de France Stage 1 (P3). Tour de France: best result P1.

=== VINGEGAARD Jonas 2023 ===
Rider: VINGEGAARD Jonas. Season: 2023. Team: Jumbo-Visma. Competed in 4 races. Results: 6 wins, 15 podiums, 22 top-10 finishes, 0 DNFs/DNS. Won: 2023 Criterium du Dauphine Stage 5, 2023 Criterium du Dauphine Stage 7, 2023 La Vuelta Ciclista a Espana Stage 13, 2023 La Vuelta Ciclista a Espana Stage 16, 2023 Paris-Nice Stage 3. Other notable results: 2023 Criterium du Dauphine Stage 4 (P2), 2023 Criterium du Dauphine Stage 8 (P2), 2023 La Vuelta Ciclista a Espana Stage 17 (P2). Tour de France: best result P1.

=== 

In [7]:
print("Generating course documents...")
course_docs = []
for _, row in course_data.iterrows():
    text = serialize_course_doc(row)
    if text.strip():
        course_docs.append({
            "id": row['Race Name'],
            "text": text,
            "metadata": {
                "race_name": row['Race Name'],
                "year": int(row.get('Year', 0)),
                "race": str(row.get('Race', '')),
                "stage": str(row.get('Stage', '')),
                "distance": float(row['Distance']) if pd.notna(row.get('Distance')) else None,
                "vertical_gain": float(row['Vertical Gain']) if pd.notna(row.get('Vertical Gain')) else None,
                "doc_type": "course_profile"
            }
        })

print(f"Generated {len(course_docs):,} course documents")

# Generate rider season documents
print("\nGenerating rider season documents...")
rider_docs = []
rider_seasons = merged_df.groupby(['Name', 'Year_results']).size().reset_index()

for _, row in rider_seasons.iterrows():
    text = serialize_rider_doc(row['Name'], row['Year_results'], merged_df)
    if text.strip():
        rider_docs.append({
            "id": f"{row['Name']}_{row['Year_results']}",
            "text": text,
            "metadata": {
                "rider_name": row['Name'],
                "year": int(row['Year_results']),
                "doc_type": "rider_season"
            }
        })

print(f"Generated {len(rider_docs):,} rider season documents")
print(f"\nTotal documents to embed: {len(course_docs) + len(rider_docs):,}")

Generating course documents...
Generated 963 course documents

Generating rider season documents...
Generated 5,835 rider season documents

Total documents to embed: 6,798


In [8]:
def embed_and_upsert(
    client: QdrantClient,
    model: SentenceTransformer,
    docs: list[dict],
    collection_name: str,
    batch_size: int = 64
):
    # Create collection if it doesn't exist
    existing = [c.name for c in client.get_collections().collections]
    if collection_name in existing:
        client.delete_collection(collection_name)
        print(f"Deleted existing collection: {collection_name}")

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=model.get_sentence_embedding_dimension(),
            distance=Distance.COSINE
        )
    )
    print(f"Created collection: {collection_name}")

    total = len(docs)
    total_upserted = 0

    for i in range(0, total, batch_size):
        batch = docs[i:i + batch_size]
        texts = [d['text'] for d in batch]

        embeddings = model.encode(texts, show_progress_bar=False)

        points = [
            PointStruct(
                id=i + j,
                vector=embedding.tolist(),
                payload={
                    "text": doc['text'],
                    "doc_id": doc['id'],
                    **doc['metadata']
                }
            )
            for j, (doc, embedding) in enumerate(zip(batch, embeddings))
        ]

        # Upsert each batch immediately rather than accumulating
        client.upsert(collection_name=collection_name, points=points)
        total_upserted += len(points)

        if (i // batch_size) % 5 == 0:
            print(f"  Upserted {total_upserted:,}/{total:,} documents")

    print(f"Done: {total_upserted:,} points upserted into '{collection_name}'")
    return total_upserted

# Embed course profiles
print("=== Embedding course profiles ===")
n_course = embed_and_upsert(client, model, course_docs, "course_profiles")

print(f"\nDone: {n_course:,} course profiles indexed")

=== Embedding course profiles ===


Created collection: course_profiles
  Upserted 64/963 documents
  Upserted 384/963 documents
  Upserted 704/963 documents
  Upserted 963/963 documents
Done: 963 points upserted into 'course_profiles'

Done: 963 course profiles indexed


In [9]:
print("=== Embedding rider season profiles ===")
n_riders = embed_and_upsert(client, model, rider_docs, "rider_seasons")
print(f"\nDone: {n_riders:,} rider season profiles indexed")

# Summary
print("\n=== Qdrant Collections ===")
for col in client.get_collections().collections:
    info = client.get_collection(col.name)
    print(f"  {col.name}: {info.points_count:,} points, "
          f"{info.config.params.vectors.size}d vectors")

=== Embedding rider season profiles ===
Created collection: rider_seasons
  Upserted 64/5,835 documents
  Upserted 384/5,835 documents
  Upserted 704/5,835 documents
  Upserted 1,024/5,835 documents
  Upserted 1,344/5,835 documents
  Upserted 1,664/5,835 documents
  Upserted 1,984/5,835 documents
  Upserted 2,304/5,835 documents
  Upserted 2,624/5,835 documents
  Upserted 2,944/5,835 documents
  Upserted 3,264/5,835 documents
  Upserted 3,584/5,835 documents
  Upserted 3,904/5,835 documents
  Upserted 4,224/5,835 documents
  Upserted 4,544/5,835 documents
  Upserted 4,864/5,835 documents
  Upserted 5,184/5,835 documents
  Upserted 5,504/5,835 documents
  Upserted 5,824/5,835 documents
Done: 5,835 points upserted into 'rider_seasons'

Done: 5,835 rider season profiles indexed

=== Qdrant Collections ===
  course_profiles: 963 points, 384d vectors
  rider_seasons: 5,835 points, 384d vectors


In [10]:
def semantic_search(
    query: str,
    collection: str,
    top_k: int = 5,
) -> list[dict]:
    """Run a semantic search query against a Qdrant collection."""
    query_vector = model.encode(query).tolist()

    results = client.query_points(
        collection_name=collection,
        query=query_vector,
        limit=top_k,
        with_payload=True
    )

    return [
        {
            "score": round(r.score, 4),
            "doc_id": r.payload.get("doc_id"),
            "text": r.payload.get("text", "")[:300]
        }
        for r in results.points
    ]

# Test queries
queries = [
    ("What are the hardest mountain stages?", "course_profiles"),
    ("Cobblestone classics races", "course_profiles"),
    ("Flat sprint stages", "course_profiles"),
    ("Who won the Tour de France in 2023?", "rider_seasons"),
    ("Best climbers in 2022", "rider_seasons"),
]

for query, collection in queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"Collection: {collection}")
    print("-" * 60)
    results = semantic_search(query, collection)
    for r in results:
        print(f"  [{r['score']:.4f}] {r['doc_id']}")
        print(f"           {r['text'][:150]}...")
        print()


Query: 'What are the hardest mountain stages?'
Collection: course_profiles
------------------------------------------------------------
  [0.4786] 2019 Paris-Nice Stage 7
           Race: 2019 Paris-Nice Stage 7. Distance: 182.1 km. This is a high mountain stage with 4660m of vertical gain, reaching a maximum elevation of 1606m an...

  [0.4632] 2022 Criterium du Dauphine Stage 3
           Race: 2022 Criterium du Dauphine Stage 3. Distance: 169.3 km. This is a hilly stage with significant climbing with 2660m of vertical gain, reaching a ...

  [0.4627] 2021 Paris-Nice Stage 6
           Race: 2021 Paris-Nice Stage 6. Distance: 201.7 km. This is a hilly stage with significant climbing with 3210m of vertical gain, reaching a maximum ele...

  [0.4555] 2023 Criterium du Dauphine Stage 8
           Race: 2023 Criterium du Dauphine Stage 8. Distance: 152.8 km. This is a high mountain stage with 4140m of vertical gain, reaching a maximum elevation ...

  [0.4546] 2022 Criterium du Dauphine

In [ ]:
from qdrant_client import QdrantClient
client = QdrantClient(url="http://localhost:6333")

# See what collections exist
for col in client.get_collections().collections:
    info = client.get_collection(col.name)
    print(f"\nCollection: {col.name}")
    print(f"  Points:     {info.points_count:,}")
    print(f"  Dimensions: {info.config.params.vectors.size}")
    print(f"  Distance:   {info.config.params.vectors.distance}")

# Peek at actual stored documents
results = client.scroll(
    collection_name="course_profiles",
    limit=3,
    with_payload=True,
    with_vectors=False   # don't return raw vectors — too large to read
)
for point in results[0]:
    print(f"\nID: {point.id}")
    print(f"Payload: {point.payload}")


Collection: course_profiles
  Points:     963
  Dimensions: 384
  Distance:   Cosine

Collection: rider_seasons
  Points:     5,835
  Dimensions: 384
  Distance:   Cosine

ID: 0
Payload: {'text': 'Race: 2022 Bretagne Classic - Ouest-France. Distance: 255.4 km. This is a hilly stage with significant climbing with 2630m of vertical gain, reaching a maximum elevation of 248m and a minimum elevation of 5m. Net elevation change: nanm. Surface breakdown: 178.0km of asphalt, 237.0km of road, 77.7km of paved road. Total descending: 2630m.', 'doc_id': '2022 Bretagne Classic - Ouest-France', 'race_name': '2022 Bretagne Classic - Ouest-France', 'year': 2022, 'race': 'Bretagne Classic - Ouest-France', 'stage': 'nan', 'distance': 255.37, 'vertical_gain': 2630.0, 'doc_type': 'course_profile'}

ID: 1
Payload: {'text': 'Race: 2022 Il Lombardia. Distance: 253.4 km. This is a high mountain stage with 5230m of vertical gain, reaching a maximum elevation of 1070m and a minimum elevation of 199m. Net elev

: 